<a href="https://colab.research.google.com/github/Fraanas/Big-Data/blob/main/SG_BigData_04_SparkSQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Big Data — Spark SQL
## Rozbudowane zajęcia laboratoryjne

**Tryb pracy:** teoria → przykłady → praca własna studenta  
**Temat przewodni:** Spark SQL jako warstwa analityczna nad dużymi zbiorami danych

---

## Cele zajęć
Po zajęciach student:
1. rozumie czym jest Spark SQL i jak łączy się z DataFrame API,
2. potrafi tworzyć widoki tymczasowe i wykonywać zapytania SQL,
3. rozumie agregacje, joiny, subquery i CTE,
4. potrafi stosować window functions (`RANK`, `ROW_NUMBER`),
5. zna podstawy interpretacji `EXPLAIN`,
6. potrafi porównać podejście SQL i DataFrame pod kątem czytelności i wydajności.


## Plan zajęć
1. Wprowadzenie teoretyczne  
2. Przygotowanie środowiska i danych  
3. Podstawowe zapytania Spark SQL  
4. Window functions  
5. Subquery i CTE  
6. Optymalizacja i `EXPLAIN`  
7. Praca własna studenta  
8. Zadanie porównawcze: SQL vs DataFrame


# Część 1. Wprowadzenie teoretyczne

## 1.1. Czym jest Spark SQL?

Spark SQL to moduł Apache Spark służący do pracy z danymi strukturalnymi i półstrukturalnymi.  
Pozwala korzystać z dwóch powiązanych podejść:

- **SQL** — zapytania w stylu relacyjnych baz danych,
- **DataFrame API** — programistyczna praca na kolumnach i transformacjach.

W praktyce oba podejścia często prowadzą do bardzo podobnych planów wykonania, ponieważ są tłumaczone przez ten sam silnik optymalizacji.

## 1.2. Dlaczego Spark SQL jest ważny w Big Data?

W środowisku Big Data liczy się nie tylko zapis kodu, ale także:
- skalowalność,
- optymalizacja wykonania,
- umiejętność pracy na dużych wolumenach danych,
- łączenie różnych źródeł danych,
- czytelność i utrzymywalność pipeline'u analitycznego.

Spark SQL jest atrakcyjny, ponieważ łączy prostotę języka SQL z rozproszonym wykonaniem.

## 1.3. DataFrame a SQL

W Sparku:
- **DataFrame** można traktować jak tabelę,
- **temp view** pozwala zarejestrować DataFrame pod nazwą,
- od tego momentu można wykonywać `spark.sql(...)`.

To oznacza, że można:
1. przygotować dane programistycznie,
2. udostępnić je jako widok,
3. analizować je dalej w SQL.

## 1.4. Temp view, global temp view, tabela trwała

Najczęściej w notebookach używa się:
- `createOrReplaceTempView(...)` — widok tymczasowy dostępny w bieżącej sesji Spark,
- `createOrReplaceGlobalTempView(...)` — widok globalny w przestrzeni `global_temp`.

W tych zajęciach skupiamy się na **temp view**, bo najlepiej nadaje się do pracy laboratoryjnej.

## 1.5. Jak Spark wykonuje zapytanie?

Spark nie wykonuje wszystkiego natychmiast. Najpierw buduje:
- plan logiczny,
- plan zoptymalizowany,
- plan fizyczny.

Dopiero potem uruchamia zadania na klastrze lub w trybie lokalnym.  
To dlatego analiza `EXPLAIN` jest ważna: pokazuje, **jak Spark naprawdę rozumie nasze zapytanie**.

## 1.6. Kiedy SQL, a kiedy DataFrame?

**SQL** jest wygodny, gdy:
- pytanie ma charakter relacyjny,
- chcemy łatwo pokazać logikę biznesową,
- użytkownicy znają składnię SQL.

**DataFrame API** jest wygodne, gdy:
- budujemy pipeline programistyczny,
- chcemy dynamicznie generować logikę,
- integrujemy analizę z kodem Python.

W praktyce oba podejścia warto znać równolegle.


## 1.7. Window functions — po co są potrzebne?

Funkcje okna nie agregują całej tabeli do jednego wyniku.  
Zamiast tego pozwalają liczyć metryki **w obrębie grupy**, zachowując jednocześnie szczegółowy poziom danych.

Typowe zastosowania:
- ranking produktów lub miast,
- wybór najlepszego rekordu w grupie,
- numerowanie rekordów,
- porównania między elementami w tej samej kategorii.

Na tych zajęciach skupimy się na:
- `RANK()` — ranking z możliwością remisów,
- `ROW_NUMBER()` — numeracja kolejnych rekordów bez remisów.

## 1.8. Subquery

Podzapytanie (`subquery`) to zapytanie zagnieżdżone w innym zapytaniu.  
Umożliwia budowę bardziej złożonej logiki etapami, np.:
- wybór rekordów powyżej średniej,
- filtrowanie względem wyniku pośredniego,
- porównanie z agregatem.

## 1.9. CTE (`WITH`)

CTE, czyli Common Table Expression, pozwala nazwać wynik pośredni i wykorzystać go w dalszym zapytaniu.  
Jest to bardzo przydatne, bo:
- poprawia czytelność,
- ułatwia rozbijanie skomplikowanego SQL na etapy,
- przypomina logiczne kroki pipeline'u danych.

## 1.10. Optymalizacja i `EXPLAIN`

Polecenie `EXPLAIN` pozwala zobaczyć plan wykonania zapytania.  
W praktyce interesuje nas szczególnie:
- czy pojawia się `Exchange` / `shuffle`,
- jakie joiny wybiera Spark,
- czy zapytanie ma sensowną strukturę,
- czy SQL i DataFrame prowadzą do podobnego planu.

Na końcu zajęć student porówna:
- **wynik**,
- **czas wykonania**,
- **plan wykonania**
dla tego samego problemu w SQL i w DataFrame API.


# Część 2. Konfiguracja środowiska

> Jeżeli środowisko Spark zostało już skonfigurowane wcześniej, można pominąć poniższą komórkę.  
> W wersji laboratoryjnej najprościej korzystać z `pyspark` instalowanego bezpośrednio przez `pip`.


In [1]:
# Konfiguracja środowiska Spark (wersja uproszczona do notebooka)
SPARK_VERSION = "3.5.8"

!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq
!pip -q uninstall -y dataproc-spark-connect || true
!pip -q install pyspark==3.5.8


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.8/317.8 MB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from time import perf_counter

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("BigData_Spark_SQL")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)
print("Default parallelism:", spark.sparkContext.defaultParallelism)


Spark version: 3.5.8
Default parallelism: 2


## Krótka interpretacja środowiska
- `Spark version` mówi, jaka wersja silnika została uruchomiona.
- `Default parallelism` pokazuje domyślną liczbę równoległych zadań dla wielu operacji.
- W trybie `local[*]` Spark wykorzystuje lokalne zasoby maszyny, a nie pełny klaster.


# Część 3. Przygotowanie danych

W tej części generujemy syntetyczny zbiór danych sprzedażowych.  
Dane są sztuczne, ale celowo przypominają prosty przypadek biznesowy:
- zamówienia,
- klienci,
- miasta,
- kanały sprzedaży,
- kategorie produktów,
- segment klientów.


In [3]:
import random
from datetime import datetime, timedelta

random.seed(42)

cities = ["Poznań", "Warszawa", "Wrocław", "Gdańsk", "Kraków", "Łódź"]
categories = ["Elektronika", "Dom", "Spożywcze", "Sport", "Książki"]
channels = ["online", "sklep", "mobile"]
segments = ["basic", "silver", "gold", "vip"]

rows = []
start = datetime(2025, 3, 1, 8, 0, 0)

for i in range(1, 3001):
    category = random.choice(categories)
    city = random.choice(cities)
    channel = random.choice(channels)
    segment = random.choice(segments)
    quantity = random.randint(1, 5)
    price = round(random.uniform(15, 900), 2)
    order_ts = start + timedelta(hours=random.randint(0, 720), minutes=random.randint(0, 59))
    customer_id = random.randint(1, 500)
    rows.append((i, customer_id, order_ts.strftime("%Y-%m-%d %H:%M:%S"), city, category, channel, segment, quantity, price))

columns = ["order_id", "customer_id", "order_ts", "city", "category", "channel", "segment", "quantity", "unit_price"]

sales_df = spark.createDataFrame(rows, columns)

sales_df = (
    sales_df
    .withColumn("order_ts", F.to_timestamp("order_ts"))
    .withColumn("revenue", F.round(F.col("quantity") * F.col("unit_price"), 2))
    .withColumn("order_date", F.to_date("order_ts"))
    .withColumn("year", F.year("order_ts"))
    .withColumn("month", F.month("order_ts"))
)

sales_df.show(10, truncate=False)


+--------+-----------+-------------------+--------+-----------+-------+-------+--------+----------+-------+----------+----+-----+
|order_id|customer_id|order_ts           |city    |category   |channel|segment|quantity|unit_price|revenue|order_date|year|month|
+--------+-----------+-------------------+--------+-----------+-------+-------+--------+----------+-------+----------+----+-----+
|1       |380        |2025-03-05 16:43:00|Poznań  |Elektronika|mobile |gold   |2       |212.54    |425.08 |2025-03-05|2025|3    |
|2       |259        |2025-03-10 15:14:00|Poznań  |Książki    |mobile |vip    |1       |41.37     |41.37  |2025-03-10|2025|3    |
|3       |143        |2025-03-20 11:37:00|Poznań  |Książki    |mobile |silver |5       |386.28    |1931.4 |2025-03-20|2025|3    |
|4       |173        |2025-03-10 12:48:00|Warszawa|Elektronika|mobile |vip    |3       |260.92    |782.76 |2025-03-10|2025|3    |
|5       |414        |2025-03-27 02:16:00|Poznań  |Elektronika|sklep  |basic  |3       |76

In [4]:
customers_df = spark.createDataFrame(
    [(i, random.choice(["kobieta", "mężczyzna"]), random.randint(18, 70), random.choice(segments)) for i in range(1, 501)],
    ["customer_id", "gender", "age", "loyalty_segment"]
)

stores_df = spark.createDataFrame(
    [
        ("Poznań", "west", "mall"),
        ("Warszawa", "central", "mall"),
        ("Wrocław", "west", "street"),
        ("Gdańsk", "north", "mall"),
        ("Kraków", "south", "street"),
        ("Łódź", "central", "retail_park"),
    ],
    ["city", "region", "store_type"]
)

customers_df.show(5, truncate=False)
stores_df.show(5, truncate=False)


+-----------+---------+---+---------------+
|customer_id|gender   |age|loyalty_segment|
+-----------+---------+---+---------------+
|1          |kobieta  |21 |vip            |
|2          |kobieta  |22 |vip            |
|3          |mężczyzna|66 |basic          |
|4          |kobieta  |69 |basic          |
|5          |mężczyzna|30 |vip            |
+-----------+---------+---+---------------+
only showing top 5 rows

+--------+-------+----------+
|city    |region |store_type|
+--------+-------+----------+
|Poznań  |west   |mall      |
|Warszawa|central|mall      |
|Wrocław |west   |street    |
|Gdańsk  |north  |mall      |
|Kraków  |south  |street    |
+--------+-------+----------+
only showing top 5 rows



In [5]:
sales_df.createOrReplaceTempView("sales")
customers_df.createOrReplaceTempView("customers")
stores_df.createOrReplaceTempView("stores")


## Komentarz metodyczny
Od tej chwili `sales`, `customers` i `stores` można traktować jak tabele w relacyjnej bazie danych.  
To jest kluczowy moment przejścia z DataFrame API do Spark SQL.


# Część 4. Podstawowe zapytania Spark SQL

Poniższe przykłady pokazują podstawowe operacje:
- wybór kolumn,
- filtrowanie,
- grupowanie,
- sortowanie,
- łączenie tabel.


In [6]:
spark.sql("""
SELECT order_id, city, category, channel, quantity, unit_price, revenue
FROM sales
LIMIT 10
""").show(truncate=False)


+--------+--------+-----------+-------+--------+----------+-------+
|order_id|city    |category   |channel|quantity|unit_price|revenue|
+--------+--------+-----------+-------+--------+----------+-------+
|1       |Poznań  |Elektronika|mobile |2       |212.54    |425.08 |
|2       |Poznań  |Książki    |mobile |1       |41.37     |41.37  |
|3       |Poznań  |Książki    |mobile |5       |386.28    |1931.4 |
|4       |Warszawa|Elektronika|mobile |3       |260.92    |782.76 |
|5       |Poznań  |Elektronika|sklep  |3       |765.03    |2295.09|
|6       |Łódź    |Elektronika|sklep  |4       |84.74     |338.96 |
|7       |Wrocław |Książki    |mobile |1       |55.55     |55.55  |
|8       |Warszawa|Elektronika|online |3       |416.27    |1248.81|
|9       |Warszawa|Spożywcze  |mobile |1       |554.08    |554.08 |
|10      |Warszawa|Dom        |sklep  |3       |890.73    |2672.19|
+--------+--------+-----------+-------+--------+----------+-------+



In [7]:
spark.sql("""
SELECT city, COUNT(*) AS orders, ROUND(SUM(revenue), 2) AS total_revenue
FROM sales
GROUP BY city
ORDER BY total_revenue DESC
""").show(truncate=False)


+--------+------+-------------+
|city    |orders|total_revenue|
+--------+------+-------------+
|Gdańsk  |527   |716311.4     |
|Poznań  |494   |688248.84    |
|Warszawa|527   |678806.35    |
|Kraków  |493   |678208.18    |
|Wrocław |471   |662460.53    |
|Łódź    |488   |657218.82    |
+--------+------+-------------+



In [8]:
spark.sql("""
SELECT city, category, channel,
       COUNT(*) AS orders,
       ROUND(AVG(revenue), 2) AS avg_revenue,
       ROUND(SUM(revenue), 2) AS total_revenue
FROM sales
GROUP BY city, category, channel
ORDER BY city, total_revenue DESC
""").show(30, truncate=False)


+------+-----------+-------+------+-----------+-------------+
|city  |category   |channel|orders|avg_revenue|total_revenue|
+------+-----------+-------+------+-----------+-------------+
|Gdańsk|Dom        |online |49    |1472.48    |72151.61     |
|Gdańsk|Książki    |online |52    |1215.44    |63202.75     |
|Gdańsk|Elektronika|sklep  |37    |1551.87    |57419.06     |
|Gdańsk|Spożywcze  |mobile |39    |1380.79    |53850.72     |
|Gdańsk|Sport      |mobile |40    |1341.24    |53649.59     |
|Gdańsk|Książki    |sklep  |38    |1323.82    |50305.3      |
|Gdańsk|Dom        |mobile |36    |1357.83    |48881.72     |
|Gdańsk|Elektronika|mobile |41    |1181.26    |48431.54     |
|Gdańsk|Elektronika|online |32    |1446.9     |46300.86     |
|Gdańsk|Spożywcze  |sklep  |33    |1383.84    |45666.78     |
|Gdańsk|Sport      |sklep  |31    |1440.78    |44664.11     |
|Gdańsk|Spożywcze  |online |23    |1627.02    |37421.4      |
|Gdańsk|Książki    |mobile |31    |1195.4     |37057.26     |
|Gdańsk|

In [9]:
spark.sql("""
SELECT s.city, st.region, st.store_type,
       COUNT(*) AS orders,
       ROUND(SUM(s.revenue), 2) AS total_revenue
FROM sales s
JOIN stores st
  ON s.city = st.city
GROUP BY s.city, st.region, st.store_type
ORDER BY total_revenue DESC
""").show(truncate=False)


+--------+-------+-----------+------+-------------+
|city    |region |store_type |orders|total_revenue|
+--------+-------+-----------+------+-------------+
|Gdańsk  |north  |mall       |527   |716311.4     |
|Poznań  |west   |mall       |494   |688248.84    |
|Warszawa|central|mall       |527   |678806.35    |
|Kraków  |south  |street     |493   |678208.18    |
|Wrocław |west   |street     |471   |662460.53    |
|Łódź    |central|retail_park|488   |657218.82    |
+--------+-------+-----------+------+-------------+



# Część 5. Window functions

Funkcje okna są bardzo ważne analitycznie, ponieważ pozwalają wykonywać obliczenia „wewnątrz grupy”, bez utraty poziomu szczegółowości danych.

## `RANK()`
- nadaje ranking w obrębie grupy,
- przy remisach kilka rekordów może mieć ten sam rank,
- kolejne miejsce może zostać „przeskoczone”.

## `ROW_NUMBER()`
- numeruje rekordy kolejno,
- przy remisach każdy rekord dostaje osobny numer,
- świetne do wyboru jednego rekordu „top 1” z każdej grupy.


In [10]:
spark.sql("""
SELECT city,
       category,
       ROUND(SUM(revenue), 2) AS total_revenue,
       RANK() OVER (PARTITION BY city ORDER BY SUM(revenue) DESC) AS revenue_rank
FROM sales
GROUP BY city, category
ORDER BY city, revenue_rank
""").show(50, truncate=False)


+--------+-----------+-------------+------------+
|city    |category   |total_revenue|revenue_rank|
+--------+-----------+-------------+------------+
|Gdańsk  |Elektronika|152151.46    |1           |
|Gdańsk  |Książki    |150565.31    |2           |
|Gdańsk  |Dom        |147612.22    |3           |
|Gdańsk  |Spożywcze  |136938.9     |4           |
|Gdańsk  |Sport      |129043.51    |5           |
|Kraków  |Książki    |154693.39    |1           |
|Kraków  |Sport      |139973.59    |2           |
|Kraków  |Elektronika|139174.32    |3           |
|Kraków  |Spożywcze  |131731.78    |4           |
|Kraków  |Dom        |112635.1     |5           |
|Poznań  |Elektronika|155051.18    |1           |
|Poznań  |Książki    |149026.12    |2           |
|Poznań  |Dom        |144127.5     |3           |
|Poznań  |Sport      |124647.42    |4           |
|Poznań  |Spożywcze  |115396.62    |5           |
|Warszawa|Elektronika|147594.37    |1           |
|Warszawa|Książki    |141520.56    |2           |


In [11]:
spark.sql("""
SELECT *
FROM (
    SELECT city,
           category,
           ROUND(SUM(revenue), 2) AS total_revenue,
           ROW_NUMBER() OVER (PARTITION BY city ORDER BY SUM(revenue) DESC) AS row_num
    FROM sales
    GROUP BY city, category
) t
WHERE row_num = 1
ORDER BY city
""").show(truncate=False)


+--------+-----------+-------------+-------+
|city    |category   |total_revenue|row_num|
+--------+-----------+-------------+-------+
|Gdańsk  |Elektronika|152151.46    |1      |
|Kraków  |Książki    |154693.39    |1      |
|Poznań  |Elektronika|155051.18    |1      |
|Warszawa|Elektronika|147594.37    |1      |
|Wrocław |Elektronika|142998.64    |1      |
|Łódź    |Spożywcze  |155637.27    |1      |
+--------+-----------+-------------+-------+



## Interpretacja
- W pierwszym przykładzie tworzymy ranking kategorii w każdym mieście.
- W drugim przykładzie wybieramy tylko jedną najlepszą kategorię dla każdego miasta.
- To są bardzo typowe zadania w analityce biznesowej: top kategorie, top produkty, top sklepy.


# Część 6. Subquery

Subquery przydaje się wtedy, gdy chcemy porównywać rekordy względem statystyki policzonej na innym poziomie, np.:
- większe niż średnia,
- najlepsze w ramach segmentu,
- rekordy spełniające warunek zależny od agregatu.


In [12]:
spark.sql("""
SELECT city, category, total_revenue
FROM (
    SELECT city, category, ROUND(SUM(revenue), 2) AS total_revenue
    FROM sales
    GROUP BY city, category
) s
WHERE total_revenue > (
    SELECT AVG(city_revenue)
    FROM (
        SELECT SUM(revenue) AS city_revenue
        FROM sales
        GROUP BY city
    ) t
)
ORDER BY total_revenue DESC
""").show(truncate=False)


+----+--------+-------------+
|city|category|total_revenue|
+----+--------+-------------+
+----+--------+-------------+



## Interpretacja
Najpierw liczymy przychód dla pary `(city, category)`, a następnie zostawiamy tylko te rekordy, których wartość przekracza średni przychód policzony na poziomie miast.


# Część 7. CTE (`WITH`)

CTE pozwala rozbić złożone zapytanie na czytelne etapy.  
W praktyce jest to bardzo dobra technika pisania „czystego SQL”, szczególnie gdy analiza zawiera kilka poziomów agregacji.


In [13]:
spark.sql("""
WITH city_category AS (
    SELECT city,
           category,
           COUNT(*) AS orders,
           ROUND(SUM(revenue), 2) AS total_revenue
    FROM sales
    GROUP BY city, category
),
city_totals AS (
    SELECT city,
           ROUND(SUM(total_revenue), 2) AS city_total
    FROM city_category
    GROUP BY city
)
SELECT cc.city,
       cc.category,
       cc.orders,
       cc.total_revenue,
       ct.city_total,
       ROUND(100.0 * cc.total_revenue / ct.city_total, 2) AS revenue_share_pct
FROM city_category cc
JOIN city_totals ct
  ON cc.city = ct.city
ORDER BY cc.city, revenue_share_pct DESC
""").show(50, truncate=False)


+--------+-----------+------+-------------+----------+-----------------+
|city    |category   |orders|total_revenue|city_total|revenue_share_pct|
+--------+-----------+------+-------------+----------+-----------------+
|Gdańsk  |Elektronika|110   |152151.46    |716311.4  |21.24            |
|Gdańsk  |Książki    |121   |150565.31    |716311.4  |21.02            |
|Gdańsk  |Dom        |108   |147612.22    |716311.4  |20.61            |
|Gdańsk  |Spożywcze  |95    |136938.9     |716311.4  |19.12            |
|Gdańsk  |Sport      |93    |129043.51    |716311.4  |18.02            |
|Kraków  |Książki    |114   |154693.39    |678208.18 |22.81            |
|Kraków  |Sport      |108   |139973.59    |678208.18 |20.64            |
|Kraków  |Elektronika|93    |139174.32    |678208.18 |20.52            |
|Kraków  |Spożywcze  |98    |131731.78    |678208.18 |19.42            |
|Kraków  |Dom        |80    |112635.1     |678208.18 |16.61            |
|Poznań  |Elektronika|114   |155051.18    |688248.8

## Interpretacja
W tym przykładzie:
1. tworzymy tabelę pośrednią z przychodem kategorii w miastach,
2. liczymy całkowity przychód miasta,
3. obliczamy udział kategorii w przychodzie miasta.

Takie podejście bardzo przypomina wieloetapowy pipeline raportowy.


# Część 8. Optymalizacja i `EXPLAIN`

`EXPLAIN` nie służy do liczenia wyniku, lecz do zrozumienia sposobu wykonania zapytania.  
To szczególnie ważne wtedy, gdy:
- dane rosną,
- zapytanie zawiera joiny,
- używamy agregacji,
- chcemy wykryć kosztowny shuffle.

W praktyce studenckiej warto sprawdzać:
- czy plan jest rozsądny,
- gdzie pojawiają się etapy wymiany danych,
- czy SQL i DataFrame mają podobny plan fizyczny.


In [14]:
spark.sql("""
EXPLAIN FORMATTED
SELECT city, category, ROUND(SUM(revenue), 2) AS total_revenue
FROM sales
GROUP BY city, category
ORDER BY total_revenue DESC
""").show(truncate=False)


+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [15]:
query_df = (
    sales_df
    .groupBy("city", "category")
    .agg(F.round(F.sum("revenue"), 2).alias("total_revenue"))
    .orderBy(F.desc("total_revenue"))
)

query_df.explain(mode="formatted")


== Physical Plan ==
AdaptiveSparkPlan (8)
+- Sort (7)
   +- Exchange (6)
      +- HashAggregate (5)
         +- Exchange (4)
            +- HashAggregate (3)
               +- Project (2)
                  +- Scan ExistingRDD (1)


(1) Scan ExistingRDD
Output [9]: [order_id#0L, customer_id#1L, order_ts#2, city#3, category#4, channel#5, segment#6, quantity#7L, unit_price#8]
Arguments: [order_id#0L, customer_id#1L, order_ts#2, city#3, category#4, channel#5, segment#6, quantity#7L, unit_price#8], MapPartitionsRDD[4] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Project
Output [3]: [city#3, category#4, round((cast(quantity#7L as double) * unit_price#8), 2) AS revenue#28]
Input [9]: [order_id#0L, customer_id#1L, order_ts#2, city#3, category#4, channel#5, segment#6, quantity#7L, unit_price#8]

(3) HashAggregate
Input [3]: [city#3, category#4, revenue#28]
Keys [2]: [city#3, category#4]
Functions [1]: [partial_sum(revenue#28)]
Aggregate 

## Co porównać w planie?
- czy pojawiają się podobne etapy agregacji,
- czy oba podejścia wykonują sortowanie,
- czy oba prowadzą do podobnego planu fizycznego,
- czy różni się tylko sposób zapisu, a nie sama logika wykonania.


# Część 9. SQL vs DataFrame — przykład porównawczy

W tej sekcji wykonujemy to samo zadanie na dwa sposoby:
1. jako zapytanie SQL,
2. jako transformacje DataFrame.

Następnie porównujemy:
- wynik,
- czas wykonania,
- plan wykonania.


In [16]:
sql_query = """
SELECT city, category, ROUND(SUM(revenue), 2) AS total_revenue
FROM sales
GROUP BY city, category
ORDER BY total_revenue DESC
"""

t0 = perf_counter()
sql_result = spark.sql(sql_query)
sql_rows = sql_result.collect()
t1 = perf_counter()

print("SQL rows:", len(sql_rows))
print("SQL execution time:", round(t1 - t0, 4), "seconds")
sql_result.show(20, truncate=False)


SQL rows: 30
SQL execution time: 1.5146 seconds
+--------+-----------+-------------+
|city    |category   |total_revenue|
+--------+-----------+-------------+
|Łódź    |Spożywcze  |155637.27    |
|Poznań  |Elektronika|155051.18    |
|Kraków  |Książki    |154693.39    |
|Gdańsk  |Elektronika|152151.46    |
|Gdańsk  |Książki    |150565.31    |
|Poznań  |Książki    |149026.12    |
|Gdańsk  |Dom        |147612.22    |
|Warszawa|Elektronika|147594.37    |
|Poznań  |Dom        |144127.5     |
|Łódź    |Sport      |143619.23    |
|Wrocław |Elektronika|142998.64    |
|Łódź    |Książki    |142510.86    |
|Wrocław |Sport      |142281.83    |
|Warszawa|Książki    |141520.56    |
|Kraków  |Sport      |139973.59    |
|Kraków  |Elektronika|139174.32    |
|Warszawa|Sport      |137922.53    |
|Gdańsk  |Spożywcze  |136938.9     |
|Wrocław |Książki    |132186.1     |
|Kraków  |Spożywcze  |131731.78    |
+--------+-----------+-------------+
only showing top 20 rows



In [17]:
t0 = perf_counter()
df_result = (
    sales_df
    .groupBy("city", "category")
    .agg(F.round(F.sum("revenue"), 2).alias("total_revenue"))
    .orderBy(F.desc("total_revenue"))
)
df_rows = df_result.collect()
t1 = perf_counter()

print("DataFrame rows:", len(df_rows))
print("DataFrame execution time:", round(t1 - t0, 4), "seconds")
df_result.show(20, truncate=False)


DataFrame rows: 30
DataFrame execution time: 0.7457 seconds
+--------+-----------+-------------+
|city    |category   |total_revenue|
+--------+-----------+-------------+
|Łódź    |Spożywcze  |155637.27    |
|Poznań  |Elektronika|155051.18    |
|Kraków  |Książki    |154693.39    |
|Gdańsk  |Elektronika|152151.46    |
|Gdańsk  |Książki    |150565.31    |
|Poznań  |Książki    |149026.12    |
|Gdańsk  |Dom        |147612.22    |
|Warszawa|Elektronika|147594.37    |
|Poznań  |Dom        |144127.5     |
|Łódź    |Sport      |143619.23    |
|Wrocław |Elektronika|142998.64    |
|Łódź    |Książki    |142510.86    |
|Wrocław |Sport      |142281.83    |
|Warszawa|Książki    |141520.56    |
|Kraków  |Sport      |139973.59    |
|Kraków  |Elektronika|139174.32    |
|Warszawa|Sport      |137922.53    |
|Gdańsk  |Spożywcze  |136938.9     |
|Wrocław |Książki    |132186.1     |
|Kraków  |Spożywcze  |131731.78    |
+--------+-----------+-------------+
only showing top 20 rows



## Uwaga interpretacyjna
Przy małych danych różnice czasu mogą być niewielkie i przypadkowe.  
Ważniejsze od samego czasu jest to, czy:
- wynik jest identyczny,
- plan wykonania jest podobny,
- jedna forma zapisu jest dla nas czytelniejsza.

Przy większych danych warto powtórzyć test kilkukrotnie i zadbać o podobne warunki uruchomienia.


# Część 10. Praca własna studenta

Poniżej znajdują się zadania do samodzielnego wykonania.  
Warto realizować je etapami: najpierw wersja SQL, potem interpretacja wyniku.


## Zadanie 1. Klasyczne agregacje
Napisz zapytanie SQL, które dla każdego kanału sprzedaży pokaże:
- liczbę zamówień,
- łączny przychód,
- średni przychód na zamówienie.

Wynik posortuj malejąco po łącznym przychodzie.


In [18]:
# TODO: Zadanie 1
# Wstaw tutaj własne zapytanie SQL


## Zadanie 2. Window function z `RANK()`
Dla każdego miasta wyznacz ranking kategorii według sumy przychodu.  
Zwróć:
- miasto,
- kategorię,
- sumę przychodu,
- ranking w mieście.


In [19]:
# TODO: Zadanie 2
# Użyj funkcji RANK() OVER (PARTITION BY ... ORDER BY ...)


## Zadanie 3. Window function z `ROW_NUMBER()`
Dla każdego miasta wybierz jedną kategorię o najwyższym przychodzie.  
Wynik ma zawierać tylko rekordy z numerem `1`.


In [20]:
# TODO: Zadanie 3
# Użyj ROW_NUMBER() i filtrowania po row_num = 1


## Zadanie 4. Subquery
Znajdź te miasta, których całkowity przychód jest większy niż średni przychód wszystkich miast.


In [21]:
# TODO: Zadanie 4
# Wykorzystaj podzapytanie z agregacją


## Zadanie 5. CTE (`WITH`)
Zbuduj raport udziału kanału sprzedaży w łącznym przychodzie miasta.  
Zapytanie powinno:
1. policzyć przychód kanału w mieście,
2. policzyć łączny przychód miasta,
3. wyliczyć procentowy udział kanału.


In [22]:
# TODO: Zadanie 5
# Wykorzystaj co najmniej dwa CTE


## Zadanie 6. `EXPLAIN`
Uruchom `EXPLAIN FORMATTED` dla jednego z poprzednich zapytań i odpowiedz:
- gdzie pojawia się agregacja,
- czy w planie widać sortowanie,
- czy można spodziewać się shuffle?


In [23]:
# TODO: Zadanie 6
# Uruchom EXPLAIN FORMATTED dla wybranego zapytania


## Zadanie 7. Porównanie SQL vs DataFrame (wydajność)

To zadanie jest najważniejszym elementem końcowym tych zajęć.

### Cel
Porównaj tę samą analizę wykonaną:
- w Spark SQL,
- w DataFrame API.

### Co należy zrobić?
1. Zdefiniuj ten sam problem analityczny w obu podejściach, np.:
   - przychód kategorii w miastach,
   - przychód kanałów w segmentach,
   - ranking kategorii w miastach.
2. Zmierz czas wykonania obu wersji przy użyciu `perf_counter()`.
3. Uruchom `EXPLAIN` / `explain(mode="formatted")`.
4. Porównaj:
   - wynik,
   - czytelność kodu,
   - plan wykonania,
   - obserwowany czas.

### Pytania interpretacyjne
- Czy wynik był identyczny?
- Która wersja była czytelniejsza?
- Czy plan wykonania był podobny?
- Czy zaobserwowano realną przewagę którejś metody?
- Jakie ograniczenia ma taki test w małym notebooku?


In [24]:
# TODO: Zadanie 7A — wersja SQL
# Zdefiniuj zapytanie SQL i zmierz czas wykonania


In [25]:
# TODO: Zadanie 7B — wersja DataFrame
# Zdefiniuj równoważne przekształcenie DataFrame i zmierz czas wykonania


In [26]:
# TODO: Zadanie 7C — interpretacja
# Zapisz 4–6 zdań z porównaniem SQL vs DataFrame


# Część 11. Podsumowanie

## Co było najważniejsze?
- Spark SQL jest naturalnym rozszerzeniem pracy z DataFrame.
- Window functions są kluczowe w analizie rankingowej.
- Subquery i CTE pomagają pisać bardziej złożone, ale czytelne zapytania.
- `EXPLAIN` pozwala zajrzeć do planu wykonania.
- SQL i DataFrame często prowadzą do bardzo podobnej logiki wykonania.

## Pytania końcowe
1. Kiedy SQL jest wygodniejszy niż DataFrame API?
2. Kiedy DataFrame API może być bardziej naturalne?
3. Jaką rolę pełni `EXPLAIN`?
4. Czym różni się `RANK()` od `ROW_NUMBER()`?
5. Dlaczego CTE poprawia czytelność zapytania?
